# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: 
Date: 

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/ryanmastropaolo/bootcamp_ryan_mastropaolo/homework/homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [MISS]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

1 needed file(s) missing. Put them at the paths above, relative to:
  /Users/ryanmastropaolo/bootcamp_ryan_mastropaolo/homework/homework04
If that folder looks wrong, you are running the notebook from the wrong place.


In [3]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? True


## Helpers (use or modify)

In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [5]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage answers 200 OK with a prose blob when the free daily cap (25 calls)
    # is hit, so check for the series rather than trusting the status code.
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index':'date','4. close':'close'})[['date','close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['close'] = pd.to_numeric(df_api['close'])

if not USE_ALPHA:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                         multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

v_api = validate(df_api, ['date','close']); v_api

{'missing': [], 'shape': (100, 2), 'na_total': 0}

In [6]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-alpha_symbol-AAPL_20260818-202652.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [7]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies' 
headers = {'User-Agent':'AFE-Homework/1.0'}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30); resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')
v_scrape = validate(df_scrape, list(df_scrape.columns)); v_scrape

{'missing': [], 'shape': (515, 8), 'na_total': 73}

In [8]:
_ = save_csv(df_scrape, prefix='scrape', site='example', table='markets')

Saved data/raw/scrape_site-example_table-markets_20260818-203844.csv


## Documentation

- **API Source**: [Alpha Vantage `TIME_SERIES_DAILY`](https://www.alphavantage.co/query) (`function=TIME_SERIES_DAILY`, `symbol=AAPL`, `outputsize=compact`), authenticated with `ALPHAVANTAGE_API_KEY` from `.env`. Returns the free-tier daily close price for AAPL. If the key is missing, or Alpha Vantage answers 200 OK with a prose message instead of a time series (e.g., the free-tier cap of 25 calls/day is hit), the notebook automatically falls back to `yfinance.download('AAPL', period='3mo', interval='1d')` so the pipeline never silently returns nothing. Output columns: `date`, `close`. This run used the live Alpha Vantage path (`USE_ALPHA=True`), saved to `api_source-alpha_symbol-AAPL_....csv` (100 rows, 0 missing values).

- **Scrape Source**: [Wikipedia — List of S&P 500 companies](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies), the first `<table>` found on the page. Columns: `Symbol, Security, GICS Sector, GICS Sub-Industry, Headquarters Location, Date added, CIK, Founded`. Saved to `scrape_site-example_table-markets_....csv` (515 rows, 73 missing values — mostly in `Date added`, which is blank for companies added before the index tracked that field). If the request or parse fails for any reason, the notebook falls back to a 1-row inline demo table (`Ticker`, `Price`) so the pipeline never raises.

- **Assumptions & risks**:
  - Alpha Vantage's free tier is capped at 25 requests/day and 5/minute; repeated re-runs in a short session can silently exhaust the quota (mitigated here by the `USE_ALPHA` fallback to `yfinance`, which itself depends on Yahoo's unofficial API staying stable).
  - The scraper grabs rows via `find_all('tr')` / `find_all(['th','td'])` with no table selector — it assumes the *first* table-like structure on the page is the one wanted. Wikipedia page edits (added tables, restructured headers, moved sections) would silently change what gets scraped rather than raising an error.
  - `df_scrape` columns are inferred from whatever the page's header row contains, so schema (column names/order) isn't guaranteed stable across runs; only a `Price` column (if present) is coerced to numeric — every other column stays as scraped text, including `Founded`, which mixes plain years with parenthetical notes (e.g. `"2013 (1888)"`).
  - Because the `except` branch is a broad `except Exception`, any scrape failure (network error, changed markup, blocked request) degrades silently to the 1-row demo table rather than stopping the run — worth tightening to a narrower exception type if this were used beyond a homework exercise.
  - Saved filenames record when the file was *written* (`ts()`), not the as-of date of the underlying data — Alpha Vantage's daily series and the Wikipedia table can each lag behind real-world changes by hours to days.
  - `validate()` only checks for missing required columns, `.shape`, and total NA count — it doesn't check for duplicate rows, out-of-range values, or per-column type correctness beyond the explicit numeric coercions above.

- **Confirm `.env` is not committed**: Confirmed — the repo's root `.gitignore` lists `.env`, and `git status` shows it untracked/ignored. This project is still missing an `.env.example` template, though (the file-check cell above flags it as `MISS`); add one at `homework/homework04/.env.example` containing `ALPHAVANTAGE_API_KEY=enter_here` so the notebook is runnable by anyone who clones the repo without ever seeing the real key.